# Real time Bitcoin price Analysis using Luigi – Checkpoint 1

In this notebook, we demonstrate the initial setup of a real-time Bitcoin price streaming system using the **Coinbase Pro WebSocket API** and a **Luigi-powered pipeline** for early-stage data preprocessing.

The goal of this checkpoint is to:
- Establish a live connection to stream BTC-USD prices
- Log the data in a structured CSV format
- Fetch and clean the data using Luigi pipeline tasks

Future checkpoints will include forecasting, anomaly detection, and alerting.

## Step 1: Stream BTC-USD Prices in Real Time

In [5]:
# stream_btc_prices.py
# This script connects to Coinbase's WebSocket feed and writes real-time Bitcoin prices to a CSV file.

import asyncio
import websockets
import json
import csv
import os
from datetime import datetime

class CoinbaseWebSocketCSVLogger:
    def __init__(self, product_id='BTC-USD', output_file='btc_price_log.csv'):
        self.url = 'wss://ws-feed.exchange.coinbase.com'
        self.product_id = product_id
        self.output_path = os.path.join('data', output_file)
        os.makedirs('data', exist_ok=True)
        if not os.path.exists(self.output_path):
            with open(self.output_path, mode='w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['timestamp', 'price'])

    async def connect(self):
        while True:
            try:
                async with websockets.connect(self.url) as websocket:
                    await self.subscribe(websocket)
                    await self.listen(websocket)
            except Exception as e:
                print(f'Connection error: {e}. Retrying in 5 seconds...')
                await asyncio.sleep(5)

    async def subscribe(self, websocket):
        subscribe_message = {
            'type': 'subscribe',
            'channels': [{'name': 'ticker', 'product_ids': [self.product_id]}]
        }
        await websocket.send(json.dumps(subscribe_message))

    async def listen(self, websocket):
        print('Streaming BTC prices to CSV...')
        async for message in websocket:
            data = json.loads(message)
            if data['type'] == 'ticker':
                self.log_to_csv(data)

    def log_to_csv(self, data):
        ts = data.get('time', '')
        price = data.get('price', '')
        if ts and price:
            timestamp = datetime.utcnow().isoformat()
            print(f'{timestamp} | Price: ${price}')
            with open(self.output_path, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([timestamp, price])

## Step 2: Luigi Pipeline – Fetch and Clean Tasks

In [7]:
# btc_pipeline_realtime.py
# This Luigi pipeline reads the streamed CSV, formats the data, and saves it in a cleaned CSV form.
%pip install luigi
import luigi
import pandas as pd
import json
from datetime import datetime

class FetchDataTask(luigi.Task):
    '''
    Task to convert the streamed CSV to a structured JSON file.
    '''
    date = luigi.DateParameter(default=datetime.utcnow().date())

    def output(self):
        return luigi.LocalTarget(f'data/raw_{self.date}.json')

    def run(self):
        with open('data/btc_price_log.csv') as f:
            lines = f.readlines()[1:]  # Skip header
        data = [{'timestamp': line.split(',')[0], 'price': float(line.split(',')[1])} for line in lines]
        with self.output().open('w') as f:
            json.dump(data, f)

class CleanDataTask(luigi.Task):
    '''
    Task to read the raw JSON file and output cleaned, timestamped data.
    '''
    date = luigi.DateParameter(default=datetime.utcnow().date())

    def requires(self):
        return FetchDataTask(self.date)

    def output(self):
        return luigi.LocalTarget(f'data/clean_{self.date}.csv')

    def run(self):
        with self.input().open('r') as f:
            data = json.load(f)
        df = pd.DataFrame(data)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.sort_values('timestamp')
        df.to_csv(self.output().path, index=False)

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 1.2 MB 7.8 MB/s eta 0:00:01
  Created wheel for luigi: filename=luigi-3.6.0-py3-none-any.whl size=1093785 sha256=5c88f54a881136cf1e6fc3101a6f86f586c6f94059e35579ebce516e1b7432cf
  Stored in directory: /Users/harshithamurali/Library/Caches/pip/wheels/21/a2/f6/ee756c360e845cb695b3550361206220ef7715e1c64f6034be
Successfully built luigi
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
